# ADS v2: campaign history

CRM campaign responses (Xtremepush sends, opens, clicks, bounces; workflow rows left out as notebook 08) summarised per month, SCV key, policy, client, channel, campaign and workflow node, the grain of ods.scvcampaignhistorynew: first / last time and count per interaction type, Bounced, the campaign name, and how the SCV key was linked (ScvLinkedBy). Unlinked responses are kept with a null SCV key (notebook 08 dropped them). Compared with ods.scvcampaignhistorynew in ads_v2.CampaignHistoryParity where that table exists.

Generated by `ads_build/v2/build_ads.py` from `families/*.yaml`. Event descriptions come from `seeds/ChillEpisodeTypes.csv`.

**If a cell fails**

1. Read the failing cell's header: what it builds, what it reads, its grain, key and dedupe rule.
2. Fix the cause. Most failures are a source change (a renamed or missing column) or a broken key (duplicates).
   The checks cell at the end names the table and the likely cause.
3. Re-run from the failed cell: every cell rebuilds its table completely (`create or replace`), so re-running is
   always safe. After a session restart, run the first three cells again first (parameters, setup, start log).
4. **Copy the fix back into the config** (`ads_build/v2/families/*.yaml`) and regenerate. This notebook is generated:
   a fix made only here is lost at the next generation (the drift check lists such cells).


In [ ]:
%%pyspark
# Parameters (leave blank when running by hand)
batch_id = ""


In [ ]:
%%pyspark
%run ADS_v2_00_Setup

In [ ]:
%%pyspark
ads_run_log("23_CampaignHistory", "started")


In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_ch_event                                                                    */
/*  Description : Every Xtremepush response but workflow rows, with notebook 08's keys: policy code ('N */
/*              : /A' as none), client code (else the first six characters of a nine-character policy c */
/*              : ode), channel = message type.                                                         */
/*              :                                                                                       */
/*              : Reads: ods_v2.MessageEvent                                                            */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_ch_event as
select  m.EventDateTime,
        year(m.EventDateTime) * 100 + month(m.EventDateTime)            as MonthId,
        nullif(nullif(upper(trim(m.PolicyCode)), ''), 'N/A')                               as PolicyCode,
        coalesce(nullif(upper(trim(m.ClientCode)), ''),
                 case when length(nullif(upper(trim(m.PolicyCode)), '')) = 9 then left(nullif(upper(trim(m.PolicyCode)), ''), 6) end) as ClientCode,
        m.MessageType                                                   as Channel,
        cast(m.CampaignId as string)                                    as campaign_id,
        m.CampaignName,
        m.WorkflowNode                                                  as workflow_node,
        m.InteractionType,
        m.QuoteGuidKey,
        m.EmailKey,
        m.MobileKey
from    ${ads.ods}.MessageEvent m
where   coalesce(m.MessageType, 'NULL') <> 'WORKFLOW'

In [ ]:
/******************************************************************************************************/
/*  Name        : stg.adsv2_ch_event_scv                                                                */
/*  Description : Each response's SCV key: policy code (any system), client code (Relay), MFQ quote gui */
/*              : d, then the email and the mobile as at the response time (ContactScvLink); LinkRank s */
/*              : ays which (1 policy .. 5 mobile, 9 none).                                             */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_ch_event, stg.adsv2_scv_by_policy, stg.adsv2_scv_by_customer, stg.ad */
/*              : sv2_scv_by_quote, stg.adsv2_contact_scv_link                                          */
/******************************************************************************************************/
create or replace table ${ads.stg}.adsv2_ch_event_scv as
select  e.*,
        coalesce(pl.SCV_Customer_Key, cl.SCV_Customer_Key, qs.SCV_Customer_Key, ek.SCV_Customer_Key,
                 mk.SCV_Customer_Key)                                   as SCV_Customer_Key,
        case when pl.SCV_Customer_Key is not null then 1
             when cl.SCV_Customer_Key is not null then 2
             when qs.SCV_Customer_Key is not null then 3
             when ek.SCV_Customer_Key is not null then 4
             when mk.SCV_Customer_Key is not null then 5
             else 9 end                                                 as LinkRank
from    ${ads.stg}.adsv2_ch_event e
left join ${ads.stg}.adsv2_scv_by_policy pl
        on  pl.PolicyRef = e.PolicyCode
left join ${ads.stg}.adsv2_scv_by_customer cl
        on  cl.SourceSystemId = 1 and cl.CustomerRef = e.ClientCode
left join ${ads.stg}.adsv2_scv_by_quote qs
        on  qs.SourceSystemId = 2 and qs.QuoteRef = e.QuoteGuidKey
left join ${ads.stg}.adsv2_contact_scv_link ek
        on  ek.ContactKey = e.EmailKey and ek.ContactKeyType = 'EmailKey'
        and e.EventDateTime >= ek.ValidFrom and (ek.ValidTo is null or e.EventDateTime < ek.ValidTo)
left join ${ads.stg}.adsv2_contact_scv_link mk
        on  mk.ContactKey = e.MobileKey and mk.ContactKeyType = 'PhoneKey'
        and e.EventDateTime >= mk.ValidFrom and (mk.ValidTo is null or e.EventDateTime < mk.ValidTo)

In [ ]:
/******************************************************************************************************/
/*  Name        : ads_v2.CampaignHistory                                                                */
/*  Description : One row per month, SCV key, policy, client, channel, campaign and workflow node (ods. */
/*              : scvcampaignhistorynew's grain and columns, plus campaign_name, ScvLinkedBy and the ba */
/*              : tch). Counts are responses (null when none, as before); CH_Id is a stable hash of the */
/*              :  grain (notebook 08 numbered rows, different every run).                              */
/*              :                                                                                       */
/*              : Reads: stg.adsv2_ch_event_scv                                                         */
/******************************************************************************************************/
create or replace table ${ads.target}.CampaignHistory as
select  xxhash64(cast(MonthId as string), coalesce(cast(SCV_Customer_Key as string), ''), coalesce(PolicyCode, ''),
                 coalesce(ClientCode, ''), coalesce(Channel, ''), coalesce(campaign_id, ''),
                 coalesce(workflow_node, ''))                           as CH_Id,
        MonthId,
        SCV_Customer_Key,
        PolicyCode,
        ClientCode,
        Channel,
        campaign_id,
        max(CampaignName)                                               as campaign_name,
        workflow_node,
        min(case when InteractionType = 'sent' then EventDateTime end)  as min_SentDateTime,
        max(case when InteractionType = 'sent' then EventDateTime end)  as max_SentDateTime,
        nullif(count(case when InteractionType = 'sent' then 1 end), 0) as SentCount,
        min(case when InteractionType = 'open' then EventDateTime end)  as min_OpenDateTime,
        max(case when InteractionType = 'open' then EventDateTime end)  as max_OpenDateTime,
        nullif(count(case when InteractionType = 'open' then 1 end), 0) as OpenCount,
        min(case when InteractionType = 'click' then EventDateTime end) as min_ClickDateTime,
        max(case when InteractionType = 'click' then EventDateTime end) as max_ClickDateTime,
        nullif(count(case when InteractionType = 'click' then 1 end), 0) as ClickCount,
        min(case when InteractionType = 'bounce' then EventDateTime end) as min_BounceDateTime,
        max(case when InteractionType = 'bounce' then EventDateTime end) as max_BounceDateTime,
        max(case when InteractionType = 'bounce' then 1 else 0 end)     as Bounced,
        case min(LinkRank) when 1 then 'policy' when 2 then 'client' when 3 then 'quote'
                           when 4 then 'email' when 5 then 'mobile' else 'none' end as ScvLinkedBy,
        '${ads.batch_id}'                                               as AdsBatchId
from    ${ads.stg}.adsv2_ch_event_scv
group by MonthId, SCV_Customer_Key, PolicyCode, ClientCode, Channel, campaign_id, workflow_node

In [ ]:
%%pyspark
# ads_v2.CampaignHistoryParity: Per month: rows, linked rows and sent / open / click totals in ads_v2.CampaignHistory vs ods.scvcampaignhistorynew, and the rows of each linked on the same month + policy / client + channel + campaign + node (Fabric only).
#
# Reads: ads_v2.CampaignHistory
old_table = globals().get("campaign_history_source") or "ods.scvcampaignhistorynew"
if not spark.catalog.tableExists(old_table):
    print(f"{old_table} not found: parity skipped")
else:
    spark.sql(f"""
        create or replace table {target_schema}.CampaignHistoryParity as
        select  coalesce(o.MonthId, n.MonthId)                         as MonthId,
                coalesce(o.Rows_, 0)                                   as OldRows,
                coalesce(n.Rows_, 0)                                   as NewRows,
                coalesce(n.Linked, 0)                                  as NewLinkedRows,
                coalesce(o.Sent, 0)                                    as OldSent,
                coalesce(n.Sent, 0)                                    as NewSent,
                coalesce(n.LinkedSent, 0)                              as NewLinkedSent,
                coalesce(o.Opened, 0)                                  as OldOpen,
                coalesce(n.Opened, 0)                                  as NewOpen,
                coalesce(o.Clicked, 0)                                 as OldClick,
                coalesce(n.Clicked, 0)                                 as NewClick,
                coalesce(s.SameKeyRows, 0)                             as SameKeyRows,
                coalesce(s.SameScv, 0)                                 as SameKeySameScv,
                '{batch_id}'                                           as AdsBatchId
        from    (select MonthId, count(*) as Rows_, sum(SentCount) as Sent, sum(OpenCount) as Opened,
                        sum(ClickCount) as Clicked
                 from {old_table} group by MonthId) o
        full join (select MonthId, count(*) as Rows_, count(SCV_Customer_Key) as Linked, sum(SentCount) as Sent,
                          sum(case when SCV_Customer_Key is not null then SentCount end) as LinkedSent,
                          sum(OpenCount) as Opened, sum(ClickCount) as Clicked
                   from {target_schema}.CampaignHistory group by MonthId) n
                on n.MonthId = o.MonthId
        left join (select o.MonthId, count(*) as SameKeyRows,
                          sum(case when o.SCV_Customer_Key = n.SCV_Customer_Key then 1 else 0 end) as SameScv
                   from {old_table} o
                   join {target_schema}.CampaignHistory n
                          on  n.MonthId = o.MonthId and n.campaign_id = cast(o.campaign_id as string)
                          and n.PolicyCode <=> upper(trim(o.PolicyCode)) and n.ClientCode <=> upper(trim(o.ClientCode))
                          and n.Channel <=> o.Channel and n.workflow_node <=> o.workflow_node
                   group by o.MonthId) s
                on s.MonthId = coalesce(o.MonthId, n.MonthId)""")
    spark.sql(f"select * from {target_schema}.CampaignHistoryParity where MonthId >= 202101 order by MonthId").show(100, truncate=False)


In [ ]:
%%pyspark
# campaign history summary: Rows and responses by how the SCV key was linked (printed).
#
# Reads: ads_v2.CampaignHistory
spark.sql(f"""
    select  ScvLinkedBy, count(*) as Rows_, sum(SentCount) as Sent, sum(OpenCount) as Opened,
            round(100 * count(*) / sum(count(*)) over (), 1) as PctRows
    from    {target_schema}.CampaignHistory
    group by ScvLinkedBy
    order by Rows_ desc""").show(truncate=False)


## Checks


In [ ]:
%%pyspark
# Checks for the tables this notebook built. All are checked and logged to AdsBuildLog; the cell then fails if
# any check failed, naming the table and the likely cause.
import json, re
from functools import reduce
from pyspark.sql import functions as F
SPEC = json.loads(r"""[{"table": "CampaignHistory", "bk": ["MonthId", "SCV_Customer_Key", "PolicyCode", "ClientCode", "Channel", "campaign_id", "workflow_node"], "sk": null, "nullable": ["SCV_Customer_Key", "PolicyCode", "ClientCode", "Channel", "campaign_id", "workflow_node"], "allow": []}]""")
PII_NAME = re.compile(r"(e_?mail|forename|first_?name|surname|last_?name|full_?name|second_?name|phone|mobile|telephone|post_?code|eircode(?!_?area)|address_?line|risk_?address|postal_?address|dob$|date_?of_?birth|licen[cs]e_?num|registration_?no|reg_?number|registration_number|^ani$|^dnis$|originatingdnis|^payer_|iban|^bic|^contact$|rendered_?message)", re.I)
EMAIL_RE = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
MOBILE_RE = r"(^|[^0-9A-Za-z-])(\+?353 ?|0)8[35-9][ -]?[0-9]{3}[ -]?[0-9]{4}([^0-9A-Za-z-]|$)"   # stands alone, not inside an id
rows, failures = [], []
for s in SPEC:
    t = f"{target_schema}.{s['table']}"
    df = spark.table(t)
    bk = s["bk"]
    must = [c for c in bk if c not in s["nullable"]]
    agg = df.agg(F.count(F.lit(1)).alias("n"),
                 F.countDistinct(F.struct(*bk)).alias("d_bk"),
                 F.sum(F.when(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in must] or [F.lit(False)]), 1).otherwise(0)).alias("null_bk"),
                 *( [F.countDistinct(s["sk"]).alias("d_sk")] if s["sk"] else [] )).first()
    pii_cols = [c for c in df.columns if PII_NAME.search(c) and not c.endswith("Key") and c not in s["allow"]]
    text_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]   # hashes never match
    hits = {}
    if text_cols:
        found = df.select(*text_cols).limit(1000000).agg(
            *[F.sum((F.col(c).rlike(EMAIL_RE) | F.col(c).rlike(MOBILE_RE)).cast("int")).alias(c) for c in text_cols]).first().asDict()
        hits = {c: v for c, v in found.items() if v}
    problems, advice = [], []
    keys = " + ".join(bk)
    if agg["n"] == 0:
        problems.append("empty")
        advice.append("the table is empty: its source is empty, or a filter (cut-off date, join) removed every row")
    if agg["d_bk"] != agg["n"]:
        problems.append(f"{agg['n'] - agg['d_bk']:,} duplicate business keys")
        advice.append(f"several rows share a key ({keys}): the source sent the same record more than once, or the "
                      f"dedupe rule in the cell no longer holds. Examples: select {', '.join(bk)}, count(*) from {t} "
                      f"group by {', '.join(bk)} having count(*) > 1")
    if agg["null_bk"]:
        problems.append(f"{agg['null_bk']:,} null business keys")
        advice.append(f"key columns ({', '.join(must)}) are empty on some rows: if that is legitimate for a column, add it "
                      f"to key_nullable for this table in the config; otherwise the source changed")
    if s["sk"] and agg["d_sk"] != agg["n"]:
        problems.append("surrogate key not unique")
        advice.append("the surrogate key repeats: the business key is not unique (see above)")
    if pii_cols:
        problems.append(f"PII-named columns {pii_cols}")
        advice.append("columns named like personal data: hash them (HASH_*) or leave them out in the config")
    if hits:
        problems.append(f"email/mobile values in {hits}")
        advice.append("emails / mobile numbers inside these columns: wrap the column in REDACT() (or hash it) in the config")
    rows.append((batch_id, s["table"], agg["n"], agg["n"] - agg["d_bk"], agg["null_bk"], "; ".join(problems) or "ok"))
    print(f"{s['table']:<28} {agg['n']:>13,}  {'; '.join(problems) or 'ok'}")
    if problems:
        failures.append(f"{s['table']}: " + " | ".join(advice))
log = spark.createDataFrame(rows, "AdsBatchId string, TableName string, RowCount long, DuplicateKeys long, NullKeys long, Result string")
log.withColumn("CheckedAt", F.current_timestamp()).write.mode("append").saveAsTable(f"{target_schema}.AdsBuildLog")
if failures:
    ads_run_log("23_CampaignHistory", "failed", "; ".join(f.split(":")[0] for f in failures))
    raise AssertionError("ads checks failed:\n- " + "\n- ".join(failures))
ads_run_log("23_CampaignHistory", "succeeded")


In [ ]:
%%pyspark
ads_run_log("23_CampaignHistory", "succeeded")
